In [0]:
# ── FILE PATH ─────────────────────────────────────────────────────────────────
INPUT_PATH  = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/lbe_model_input.csv"   # output from Script 1
OUTPUT_DIR  = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_LBE"                  # directory for all plot outputs
RESULTS_PKL = r"/Volumes/data_exploration_hub/marketing_hub/model_data/Final_Data/MMM_LBE/lbe_mmm_results.pkl"  # pickled model state
# ─────────────────────────────────────────────────────────────────────────────

# ── MODEL CONFIG — adjust if needed ──────────────────────────────────────────
TRAIN_CUTOFF_YEAR = 2025   # train on years ≤ this; val = last N_VAL weeks
N_VAL             = 12     # weeks held out for validation
DRAWS             = 2000   # posterior draws per chain
TUNE              = 2000   # tuning steps
CHAINS            = 4      # keep at 1 for single-core; set 2+ on multi-core
CORES             = 4
TARGET_ACCEPT     = 0.95
RANDOM_SEED       = 42
# ─────────────────────────────────────────────────────────────────────────────

In [0]:
%pip install pymc arviz numpyro jax jaxlib pytensor scikit-learn openpyxl --quiet

In [0]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as ss

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [0]:
# ── BRAND PALETTE ─────────────────────────────────────────────────────────────
CPX_NAVY  = "#002B5C"
CPX_RED   = "#E31837"
CPX_GOLD  = "#F5A800"
CPX_DGREY = "#4A4A4A"
CPX_LGREY = "#F0F2F5"

def set_style():
    plt.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white",
        "axes.edgecolor": "#CCCCCC", "grid.color": "#EEEEEE",
        "grid.linestyle": "-", "axes.grid": True,
        "font.family": "sans-serif", "font.size": 10,
        "axes.titlesize": 12, "axes.titleweight": "bold",
        "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
        "legend.fontsize": 9, "legend.framealpha": 0.9,
    })

set_style()


In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1. DATA LOADING & PREP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print("=" * 65)
print("  LBE MMM — Step 2: Model Training & Diagnostics")
print("=" * 65)
print("\n[1/7] Loading data...")

df_raw = pd.read_csv(INPUT_PATH, parse_dates=["week_date"])
df = df_raw[df_raw["year"] <= TRAIN_CUTOFF_YEAR].copy().reset_index(drop=True)
df["log_rev"] = np.log(df["LBE_Revenue"])
df["t"]       = np.arange(len(df), dtype=float)

n_val    = N_VAL
df_train = df.iloc[:-n_val].copy().reset_index(drop=True)
df_val   = df.iloc[-n_val:].copy().reset_index(drop=True)

print(f"      Full dataset:  {len(df)} weeks  "
      f"({df['week_date'].min().date()} → {df['week_date'].max().date()})")
print(f"      Train:         {len(df_train)} weeks  "
      f"({df_train['week_date'].min().date()} → {df_train['week_date'].max().date()})")
print(f"      Validation:    {len(df_val)} weeks  "
      f"({df_val['week_date'].min().date()} → {df_val['week_date'].max().date()})")

In [0]:
# ── NORMALISATION (fit on train only) ─────────────────────────────────────────
y_mean   = df_train["log_rev"].mean()
y_std    = df_train["log_rev"].std()
aw_max   = df_train["Awareness"].max()
co_max   = df_train["Consideration"].max()
t_max    = df_train["t"].max()
temp_mu  = df_train["national_mean_temp"].mean()
temp_sd  = df_train["national_mean_temp"].std()

CAL_COLS = [
    "is_new_year_wk", "is_march_break", "is_march_adj",
    "is_pre_xmas", "is_summer_peak",
    "is_back_to_school", "is_spring_lull",
    "holiday", "long_weekend",
]

# Informed directional priors for calendar features
# (derived from empirical WOY revenue analysis — see EDA plot)
CAL_MU    = np.array([ 0.40,  0.40,  0.10,  0.30,  0.15, -0.20, -0.15,  0.10,  0.05])
CAL_SIGMA = np.array([ 0.15,  0.15,  0.10,  0.15,  0.10,  0.10,  0.10,  0.10,  0.10])

CAL_DISPLAY = [
    "New Year Wk", "March Break", "March Adj.",
    "Pre-Xmas", "Summer Peak",
    "Back-to-School", "Spring Lull",
    "Stat Holiday", "Long Weekend",
]

def prep(d: pd.DataFrame) -> dict:
    return {
        "y":       ((d["log_rev"] - y_mean) / y_std).values.astype("float64"),
        "aw":      (d["Awareness"].values    / aw_max).astype("float64"),
        "co":      (d["Consideration"].values / co_max).astype("float64"),
        "temp":    ((d["national_mean_temp"].values - temp_mu) / temp_sd).astype("float64"),
        "cal":     d[CAL_COLS].values.astype("float64"),
        "t":       (d["t"].values / t_max).astype("float64"),
    }

tr = prep(df_train)
va = prep(df_val)

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 2. EDA PLOT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[2/7] Generating EDA plot...")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("LBE MMM — Exploratory Data Analysis", fontsize=14,
             fontweight="bold", color=CPX_NAVY)

# — 2a: Average revenue by week of year —
ax = axes[0, 0]
woy_avg = df.groupby("week_of_year")["LBE_Revenue"].mean() / 1e6
ax.bar(woy_avg.index, woy_avg.values, color=CPX_NAVY, alpha=0.7, width=0.85)
ax.axhline(woy_avg.mean(), color=CPX_RED, lw=1.5, ls="--",
           label=f"Mean ${woy_avg.mean():.2f}M")
for wk, label in [(1, "New Year"), (11, "March Bk"), (33, "Summer Pk"), (50, "Pre-Xmas")]:
    if wk in woy_avg.index:
        ax.axvline(wk, color=CPX_GOLD, lw=1, ls=":", alpha=0.8)
        ax.text(wk + 0.5, woy_avg.max() * 0.95, label, fontsize=7,
                color=CPX_GOLD, rotation=90, va="top")
ax.set_title("Average Weekly Revenue by Week of Year")
ax.set_xlabel("Week of Year"); ax.set_ylabel("Revenue ($M)"); ax.legend()

# — 2b: Revenue time series —
ax = axes[0, 1]
ax.plot(df["week_date"], df["LBE_Revenue"] / 1e6, color=CPX_NAVY, lw=1.2, alpha=0.85)
ax.fill_between(df["week_date"], df["LBE_Revenue"] / 1e6, alpha=0.12, color=CPX_NAVY)
for yr in [2024, 2025]:
    ax.axvline(pd.Timestamp(f"{yr}-01-01"), color=CPX_DGREY, lw=0.8, ls="--", alpha=0.5)
ax.axvline(df_val["week_date"].values[0], color=CPX_RED, lw=1.5, ls="--",
           label="Validation start")
ax.set_title("Weekly LBE Revenue 2023–2025"); ax.set_ylabel("Revenue ($M)"); ax.legend()

# — 2c: Partial cross-correlation (controlling for school calendar & temp) —
ax = axes[1, 0]
from numpy.linalg import lstsq
school_map = dict(zip(
    [1, 11, 10, 12, 13, 32, 33, 34, 36, 37, 38, 39, 40, 41, 49, 50],
    [3,  2,  1,  1,  1,  1,  1,  1, -1, -1, -1, -1, -1, -1,  1,  1]
))
school_num = df["week_of_year"].map(school_map).fillna(0).values
X_ctrl = np.column_stack([school_num, df["national_mean_temp"].values, np.ones(len(df))])
y_arr  = df["LBE_Revenue"].values
cy, _, _, _ = lstsq(X_ctrl, y_arr, rcond=None); y_resid = y_arr - X_ctrl @ cy
lags = range(9); aw_r, co_r = [], []
for lag in lags:
    for col, lst in [("Awareness", aw_r), ("Consideration", co_r)]:
        x = df[col].shift(lag).fillna(0).values
        cx, _, _, _ = lstsq(X_ctrl, x, rcond=None)
        lst.append(np.corrcoef(x - X_ctrl @ cx, y_resid)[0, 1])
w = 0.35; xp = np.array(list(lags))
ax.bar(xp - w/2, aw_r, w, label="Awareness",     color=CPX_NAVY, alpha=0.85)
ax.bar(xp + w/2, co_r, w, label="Consideration", color=CPX_RED,  alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
peak = int(np.argmax(aw_r))
ax.annotate(f"Peak L{peak}", xy=(peak - w/2, aw_r[peak]),
            xytext=(peak + 1, aw_r[peak] + 0.03),
            arrowprops=dict(arrowstyle="->", color=CPX_NAVY), fontsize=8, color=CPX_NAVY)
ax.set_title("Partial Cross-Correlation: Spend → Revenue by Lag\n"
             "(controlling for school calendar & temperature)")
ax.set_xlabel("Lag (weeks)"); ax.set_ylabel("Partial r")
ax.set_xticks(xp); ax.set_xticklabels([f"L{i}" for i in lags]); ax.legend()

# — 2d: Revenue by calendar period —
ax = axes[1, 1]
cal_labels = ["School-in\n(baseline)", "Summer\nPeak", "March\nBreak",
              "Pre-\nXmas", "New Year\nWeek", "Back-to-\nSchool", "Spring\nLull"]
cal_masks = [
    (df["is_new_year_wk"] == 0) & (df["is_march_break"] == 0) & (df["is_march_adj"] == 0) &
    (df["is_pre_xmas"] == 0) & (df["is_summer_peak"] == 0) &
    (df["is_back_to_school"] == 0) & (df["is_spring_lull"] == 0),
    df["is_summer_peak"] == 1, df["is_march_break"] == 1,
    df["is_pre_xmas"] == 1,   df["is_new_year_wk"] == 1,
    df["is_back_to_school"] == 1, df["is_spring_lull"] == 1,
]
means = [df.loc[m, "LBE_Revenue"].mean() / 1e6 for m in cal_masks]
colors_bar = [CPX_DGREY, CPX_NAVY, CPX_NAVY, CPX_GOLD, CPX_GOLD, CPX_RED, CPX_RED]
bars = ax.barh(cal_labels, means, color=colors_bar, alpha=0.8)
ax.axvline(means[0], color=CPX_DGREY, lw=1, ls="--", alpha=0.7, label="Baseline")
for bar, v in zip(bars, means):
    ax.text(v + 0.03, bar.get_y() + bar.get_height() / 2,
            f"${v:.2f}M", va="center", fontsize=8)
ax.set_title("Average Weekly Revenue by Calendar Period")
ax.set_xlabel("Revenue ($M)"); ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "01_eda.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"      Saved: {OUTPUT_DIR}/01_eda.png")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3. MODEL DEFINITION & TRAINING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[3/7] Training model...")

def adstock_pt(x, decay, max_lag: int = 10):
    """Geometric adstock transformation in PyTensor."""
    lags = pt.arange(max_lag, dtype="float64")
    w = decay ** lags
    w = w / w.sum()
    n = x.shape[0]
    out = pt.zeros((n,))
    for l in range(max_lag):
        out = out + w[l] * pt.concatenate([pt.zeros((l,)), x[:n - l]])
    return out

aw_t  = pt.as_tensor_variable(tr["aw"])
co_t  = pt.as_tensor_variable(tr["co"])
n_cal = len(CAL_COLS)

with pm.Model() as mmm:
    # ── Adstock decay priors ───────────────────────────────────────────────
    # Awareness: empirical lag analysis shows peak response at 4–6 weeks →
    #            slow decay; prior centred on 0.65 (Beta(6.5, 3.5))
    # Consideration: near-zero partial correlations at all lags →
    #                faster decay; prior centred on 0.30 (Beta(3, 7))
    aw_decay = pm.Beta("aw_decay", alpha=6.5, beta=3.5)
    co_decay = pm.Beta("co_decay", alpha=3.0, beta=7.0)

    aw_ads = adstock_pt(aw_t, aw_decay)
    co_ads = adstock_pt(co_t, co_decay)

    # ── Saturation: log1p (concave diminishing returns, no divergence) ─────
    aw_scale = pm.HalfNormal("aw_scale", sigma=2.0)
    co_scale = pm.HalfNormal("co_scale", sigma=1.0)   # tighter — weak signal

    aw_sat = pt.log1p(aw_ads * aw_scale)
    co_sat = pt.log1p(co_ads * co_scale)

    # ── Media coefficients (must be positive) ─────────────────────────────
    beta_aw = pm.HalfNormal("beta_aw", sigma=0.8)
    beta_co = pm.HalfNormal("beta_co", sigma=0.3)   # tighter prior — weak signal

    # ── Controls ──────────────────────────────────────────────────────────
    intercept   = pm.Normal("intercept",   mu=0,    sigma=1.0)
    beta_trend  = pm.Normal("beta_trend",  mu=0.2,  sigma=0.2)  # ~$96K/yr uplift
    beta_temp   = pm.Normal("beta_temp",   mu=-0.2, sigma=0.15) # cold → more indoor traffic
    beta_cal    = pm.Normal("beta_cal", mu=CAL_MU, sigma=CAL_SIGMA, shape=n_cal)

    cal_eff = pt.dot(pt.as_tensor_variable(tr["cal"]), beta_cal)

    mu = (intercept
          + beta_aw   * aw_sat
          + beta_co   * co_sat
          + beta_trend * tr["t"]
          + beta_temp  * tr["temp"]
          + cal_eff)

    sigma = pm.HalfNormal("sigma", sigma=0.3)
    pm.Normal("y", mu=mu, sigma=sigma, observed=tr["y"])

    # Prior predictive
    prior = pm.sample_prior_predictive(draws=500, random_seed=RANDOM_SEED)

    # Posterior
    idata = pm.sample(
        draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
        target_accept=TARGET_ACCEPT,
        progressbar=True,
        random_seed=RANDOM_SEED,
    )

    # Posterior predictive
    pm.sample_posterior_predictive(idata, extend_inferencedata=True,
                                   random_seed=RANDOM_SEED)

divs = int(idata.sample_stats["diverging"].values.sum())
print(f"\n      Divergences: {divs}")
if divs > 10:
    print("      ⚠  More than 10 divergences — consider raising target_accept to 0.95.")

# ── Extract posterior draws ────────────────────────────────────────────────
post    = idata.posterior
draws   = {k: post[k].values.flatten() for k in post.data_vars}
n_draws = len(draws["beta_aw"])
bc      = post["beta_cal"].values.reshape(-1, n_cal)

print("\n      Posterior summary (key parameters):")
summ = az.summary(idata, var_names=[
    "aw_decay", "co_decay", "aw_scale", "co_scale",
    "beta_aw", "beta_co", "beta_trend", "beta_temp", "sigma",
], round_to=3)
print(summ.to_string())

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 4. PREDICTION HELPERS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def adstock_np(x: np.ndarray, decay: float, max_lag: int = 10) -> np.ndarray:
    w = decay ** np.arange(max_lag)
    w /= w.sum()
    n, out = len(x), np.zeros(len(x))
    for l in range(max_lag):
        for t in range(l, n):
            out[t] += w[l] * x[t - l]
    return out

def predict_revenue(data: dict, n_samples: int = 1000) -> np.ndarray:
    """Return (n_samples × T) matrix of revenue predictions."""
    idx = np.random.choice(n_draws, n_samples, replace=False)
    preds = np.zeros((n_samples, len(data["aw"])))
    for j, i in enumerate(idx):
        aw_ads = adstock_np(data["aw"], float(draws["aw_decay"][i]))
        co_ads = adstock_np(data["co"], float(draws["co_decay"][i]))
        aw_sat = np.log1p(aw_ads * float(draws["aw_scale"][i]))
        co_sat = np.log1p(co_ads * float(draws["co_scale"][i]))
        cal_eff = data["cal"] @ bc[i]
        mu = (float(draws["intercept"][i])
              + float(draws["beta_aw"][i])    * aw_sat
              + float(draws["beta_co"][i])    * co_sat
              + float(draws["beta_trend"][i]) * data["t"]
              + float(draws["beta_temp"][i])  * data["temp"]
              + cal_eff)
        preds[j] = np.exp(mu * y_std + y_mean)
    return preds

np.random.seed(RANDOM_SEED)
tr_preds = predict_revenue(tr, 1000)
va_preds = predict_revenue(va, 1000)

y_tr  = np.exp(tr["y"] * y_std + y_mean)
y_va  = np.exp(va["y"] * y_std + y_mean)
tr_mean = tr_preds.mean(axis=0); tr_lo = np.percentile(tr_preds, 10, axis=0); tr_hi = np.percentile(tr_preds, 90, axis=0)
va_mean = va_preds.mean(axis=0); va_lo = np.percentile(va_preds, 10, axis=0); va_hi = np.percentile(va_preds, 90, axis=0)

train_mape = np.mean(np.abs((y_tr - tr_mean) / y_tr)) * 100
val_mape   = np.mean(np.abs((y_va - va_mean) / y_va)) * 100
train_r2   = 1 - np.sum((y_tr - tr_mean)**2) / np.sum((y_tr - y_tr.mean())**2)
val_r2     = 1 - np.sum((y_va - va_mean)**2) / np.sum((y_va - y_va.mean())**2)

print(f"\n      Train MAPE: {train_mape:.1f}%   R²: {train_r2:.3f}")
print(f"      Val   MAPE: {val_mape:.1f}%   R²: {val_r2:.3f}")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 5. DIAGNOSTIC PLOTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[4/7] Generating diagnostic plots...")

# ── Plot 2: Prior vs Posterior ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("LBE MMM — Prior vs Posterior Distributions",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

param_specs = [
    ("aw_decay",   "Awareness Adstock Decay",       0, 1,    "beta",     (6.5, 3.5)),
    ("co_decay",   "Consideration Adstock Decay",   0, 1,    "beta",     (3.0, 7.0)),
    ("beta_aw",    "Awareness β (media coeff)",      0, 3,    "halfnorm", (0.8,)),
    ("beta_co",    "Consideration β",                0, 2,    "halfnorm", (0.3,)),
    ("beta_temp",  "Temperature β",                 -1, 0.5,  "norm",     (-0.2, 0.15)),
    ("beta_trend", "Time Trend β",                 -0.5, 1,   "norm",     (0.2, 0.2)),
    ("aw_scale",   "Awareness Saturation Scale",    0, 6,    "halfnorm", (2.0,)),
    ("sigma",      "Residual σ",                    0, 1,    "halfnorm", (0.3,)),
]
for ax, (pname, title, xlo, xhi, dist, pargs) in zip(axes.flat, param_specs):
    pv = draws[pname]
    ax.hist(pv, bins=40, density=True, color=CPX_NAVY, alpha=0.55, label="Posterior")
    xx = np.linspace(max(xlo, pv.min() - 0.05), min(xhi, pv.max() + 0.05), 300)
    if dist == "beta":      pdf = ss.beta.pdf(xx, *pargs)
    elif dist == "halfnorm": pdf = ss.halfnorm.pdf(xx, 0, *pargs)
    else:                    pdf = ss.norm.pdf(xx, *pargs)
    ax.plot(xx, pdf, color=CPX_RED, lw=2, label="Prior")
    ax.axvline(pv.mean(), color=CPX_GOLD, lw=1.5, ls="--", label=f"μ={pv.mean():.3f}")
    ci_lo, ci_hi = np.percentile(pv, [10, 90])
    ax.axvspan(ci_lo, ci_hi, alpha=0.10, color=CPX_NAVY)
    ax.set_title(title, fontsize=9); ax.set_xlim(xlo, xhi); ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "02_prior_posterior.png"), dpi=150, bbox_inches="tight")
plt.show()

# ── Plot 3: Fit ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle("LBE MMM — Model Fit: Actual vs Predicted",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

all_dates  = pd.concat([df_train["week_date"], df_val["week_date"]]).values
all_actual = np.concatenate([y_tr, y_va])
all_pred   = np.concatenate([tr_mean, va_mean])
all_lo     = np.concatenate([tr_lo, va_lo])
all_hi     = np.concatenate([tr_hi, va_hi])
n_tr       = len(y_tr)

ax = axes[0]
ax.plot(all_dates, all_actual / 1e6, color=CPX_DGREY, lw=1.5, label="Actual", zorder=3)
ax.plot(all_dates[:n_tr], all_pred[:n_tr] / 1e6, color=CPX_NAVY, lw=1.5, label="Predicted (train)")
ax.plot(all_dates[n_tr:], all_pred[n_tr:] / 1e6, color=CPX_RED,  lw=1.5, label="Predicted (val)", ls="--")
ax.fill_between(all_dates[:n_tr], all_lo[:n_tr]/1e6, all_hi[:n_tr]/1e6, alpha=0.12, color=CPX_NAVY)
ax.fill_between(all_dates[n_tr:], all_lo[n_tr:]/1e6, all_hi[n_tr:]/1e6, alpha=0.20, color=CPX_RED)
ax.axvline(df_val["week_date"].values[0], color=CPX_RED, lw=1, ls=":", alpha=0.7)
ax.set_ylabel("Revenue ($M)"); ax.legend()
ax.set_title(f"Train MAPE: {train_mape:.1f}%   R²: {train_r2:.3f}   |   "
             f"Val MAPE: {val_mape:.1f}%   R²: {val_r2:.3f}")

ax = axes[1]
ax.scatter(y_tr / 1e6, tr_mean / 1e6, color=CPX_NAVY, alpha=0.5, s=25, label="Train")
ax.scatter(y_va / 1e6, va_mean / 1e6, color=CPX_RED,  alpha=0.8, s=40, marker="D", label="Val")
mn = min(all_actual.min(), all_pred.min()) / 1e6
mx = max(all_actual.max(), all_pred.max()) / 1e6
ax.plot([mn, mx], [mn, mx], color=CPX_DGREY, lw=1, ls="--", label="Perfect fit")
ax.set_xlabel("Actual Revenue ($M)"); ax.set_ylabel("Predicted Revenue ($M)")
ax.set_title("Actual vs Predicted Scatter"); ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "03_fit.png"), dpi=150, bbox_inches="tight")
plt.show()

In [0]:
from PIL import Image
import matplotlib.pyplot as plt

plot_paths = [
    os.path.join(OUTPUT_DIR, "02_prior_posterior.png"),
    os.path.join(OUTPUT_DIR, "03_fit.png"),
]

for path in plot_paths:
    img = Image.open(path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.show()

In [0]:
# ── Plot 4: Residual diagnostics ──────────────────────────────────────────
df_tr_res = df_train.copy()
df_tr_res["pred"]    = tr_mean
df_tr_res["resid"]   = y_tr - tr_mean
df_tr_res["pct_err"] = (y_tr - tr_mean) / y_tr * 100

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("LBE MMM — Residual Diagnostics",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

ax = axes[0, 0]
ax.bar(df_tr_res["week_date"], df_tr_res["resid"] / 1e3,
       color=np.where(df_tr_res["resid"] >= 0, CPX_NAVY, CPX_RED), alpha=0.6, width=5)
ax.axhline(0, color="black", lw=0.8)
ax.set_title("Residuals Over Time"); ax.set_ylabel("Residual ($K)")

ax = axes[0, 1]
season_order = ["winter", "transition", "spring", "summer", "fall"]
bp = ax.boxplot(
    [df_tr_res.loc[df_tr_res["season"] == s, "resid"].values / 1e3 for s in season_order],
    labels=season_order, patch_artist=True,
    medianprops=dict(color="white", lw=2),
)
for patch, c in zip(bp["boxes"], [CPX_NAVY]*2 + [CPX_DGREY] + [CPX_NAVY] + [CPX_RED]):
    patch.set_facecolor(c); patch.set_alpha(0.7)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title("Residuals by Season"); ax.set_ylabel("Residual ($K)")

ax = axes[0, 2]
for i, (yr, c) in enumerate(zip([2023, 2024, 2025], [CPX_NAVY, CPX_GOLD, CPX_RED])):
    sub = df_tr_res.loc[df_tr_res["year"] == yr, "resid"] / 1e3
    ax.bar(i, sub.mean(), color=c, alpha=0.8, label=str(yr))
    ax.errorbar(i, sub.mean(), yerr=sub.std(), color="black", capsize=5, lw=1.5)
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["2023", "2024", "2025"])
ax.set_title("Mean Residual by Year"); ax.set_ylabel("Mean Residual ($K)"); ax.legend()

ax = axes[1, 0]
df_tr_res["rev_q"] = pd.qcut(df_tr_res["LBE_Revenue"], 4,
                              labels=["Q1\n(low)", "Q2", "Q3", "Q4\n(high)"])
q_bias = [df_tr_res.loc[df_tr_res["rev_q"] == q, "resid"].mean() / 1e3
          for q in ["Q1\n(low)", "Q2", "Q3", "Q4\n(high)"]]
bars = ax.bar(["Q1\n(low)", "Q2", "Q3", "Q4\n(high)"], q_bias,
              color=[CPX_RED if v < 0 else CPX_NAVY for v in q_bias], alpha=0.8)
ax.axhline(0, color="black", lw=0.8, ls="--")
for bar, v in zip(bars, q_bias):
    ax.text(bar.get_x() + bar.get_width() / 2, v + (5 if v >= 0 else -5),
            f"{v:+.0f}K", ha="center", fontsize=8)
ax.set_title("Mean Residual by Revenue Quartile\n"
             "(note: bias cancels at annual level)")
ax.set_ylabel("Mean Residual ($K)")

ax = axes[1, 1]
resid_vals = df_tr_res["resid"].values
resid_std  = (resid_vals - resid_vals.mean()) / resid_vals.std()
n_obs      = len(resid_std)
df_t       = n_obs - 2   # degrees of freedom for studentised residuals

# Use t-distribution reference (correct for small samples with heavy tails)
(osm, osr), (slope, intercept, r) = ss.probplot(resid_std, dist=ss.t, sparams=(df_t,))
ax.scatter(osm, osr, color=CPX_NAVY, alpha=0.55, s=20, zorder=3, label="Residuals")
ref_x = np.array([osm.min(), osm.max()])
ax.plot(ref_x, slope * ref_x + intercept, color=CPX_RED, lw=1.5, label="t-dist reference")

# Annotate the 3 worst outliers
outlier_mask = np.abs(resid_std) > 2.8
for xi, yi, wr in zip(osm[outlier_mask], osr[outlier_mask],
                       df_tr_res.loc[np.abs(resid_std) > 2.8, "week_date"].values):
    ax.annotate(str(pd.Timestamp(wr).date()), xy=(xi, yi),
                xytext=(xi - 0.4, yi - 0.5), fontsize=7, color=CPX_DGREY,
                arrowprops=dict(arrowstyle="-", color=CPX_DGREY, lw=0.7))

ax.set_title("Q-Q Plot — Studentised Residuals vs t-distribution\n"
             "(3 outlier weeks are genuine one-off spikes, not systematic bias)")
ax.set_xlabel("Theoretical quantiles (t-dist)"); ax.set_ylabel("Sample quantiles")
ax.legend(fontsize=8)

ax = axes[1, 2]
ax.hist(df_tr_res["pct_err"], bins=30, color=CPX_NAVY, alpha=0.7, edgecolor="white")
ax.axvline(0, color=CPX_RED, lw=1.5, ls="--")
ax.axvline(df_tr_res["pct_err"].mean(), color=CPX_GOLD, lw=1.5,
           label=f"Mean: {df_tr_res['pct_err'].mean():+.1f}%")
ax.set_title("Distribution of % Errors (train)")
ax.set_xlabel("% Error"); ax.set_ylabel("Count"); ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "04_residuals.png"), dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# ── Plot 5: Calendar effects ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("LBE MMM — Calendar Effects (Posterior)",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

cal_means = bc.mean(axis=0)
cal_lo    = np.percentile(bc, 10, axis=0)
cal_hi    = np.percentile(bc, 95, axis=0)
colors_c  = [CPX_NAVY if m > 0 else CPX_RED for m in cal_means]

bars = ax.bar(CAL_DISPLAY, cal_means, color=colors_c, alpha=0.8)
ax.errorbar(np.arange(len(CAL_DISPLAY)), cal_means,
            yerr=[cal_means - cal_lo, cal_hi - cal_means],
            fmt="none", color=CPX_DGREY, capsize=5, lw=1.5)
for i, pm_val in enumerate(CAL_MU):
    ax.plot([i - 0.4, i + 0.4], [pm_val, pm_val], color=CPX_GOLD, lw=2, ls="--")

ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Coefficient (log-revenue scale)")
ax.set_title("Posterior mean ± 80% CI  |  Dashed = prior mean")
from matplotlib.lines import Line2D
ax.legend(handles=[
    Line2D([0], [0], color=CPX_GOLD, lw=2, ls="--", label="Prior mean"),
    Line2D([0], [0], color=CPX_DGREY, lw=1.5, label="80% CI"),
], loc="lower right")

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "05_calendar_effects.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

# ── Plot 6: Adstock decay curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("LBE MMM — Adstock Decay & Carryover by Funnel",
             fontsize=13, fontweight="bold", color=CPX_NAVY)
lags = np.arange(11)

for ax, (fname, dkey, color) in zip(axes, [
    ("Awareness",     "aw_decay", CPX_NAVY),
    ("Consideration", "co_decay", CPX_RED),
]):
    d_mean = draws[dkey].mean()
    sample_idx = np.random.choice(n_draws, 200, replace=False)
    for i in sample_idx:
        d_i = float(draws[dkey][i])
        w = d_i ** lags; w /= w.sum()
        ax.plot(lags, w, color=color, alpha=0.04, lw=1)
    w_mean = d_mean ** lags; w_mean /= w_mean.sum()
    ax.plot(lags, w_mean, color=color, lw=2.5, label=f"Mean decay = {d_mean:.2f}")
    ax.fill_between(lags, 0, w_mean, alpha=0.12, color=color)
    half_idx = int(np.where(np.cumsum(w_mean) >= 0.5)[0][0])
    ax.axvline(half_idx, color=CPX_GOLD, lw=1.5, ls="--",
               label=f"50% carryover by wk {half_idx}")
    ax.set_title(f"{fname}: Spend Weight by Weeks-After-Spend\n"
                 f"(shaded = posterior uncertainty)")
    ax.set_xlabel("Weeks after spend"); ax.set_ylabel("Relative weight")
    ax.set_xticks(lags); ax.set_xticklabels([f"Wk {l}" for l in lags], rotation=30)
    ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "06_adstock.png"), dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"      Saved plots 02–06 to {OUTPUT_DIR}/")

In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 6. REVENUE DECOMPOSITION & ROI
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[5/7] Revenue decomposition & ROI...")

N_DECOMP = 2000
idx_d = np.random.choice(n_draws, N_DECOMP, replace=False)
T = len(tr["aw"])
aw_c = np.zeros((N_DECOMP, T))
co_c = np.zeros((N_DECOMP, T))
base_c = np.zeros((N_DECOMP, T))

for j, i in enumerate(idx_d):
    aw_ads = adstock_np(tr["aw"], float(draws["aw_decay"][i]))
    co_ads = adstock_np(tr["co"], float(draws["co_decay"][i]))
    aw_sat = np.log1p(aw_ads * float(draws["aw_scale"][i]))
    co_sat = np.log1p(co_ads * float(draws["co_scale"][i]))
    cal_eff   = tr["cal"] @ bc[i]
    base_log  = (float(draws["intercept"][i])
                 + float(draws["beta_trend"][i]) * tr["t"]
                 + float(draws["beta_temp"][i])  * tr["temp"]
                 + cal_eff)
    aw_log  = float(draws["beta_aw"][i]) * aw_sat
    co_log  = float(draws["beta_co"][i]) * co_sat
    total_log = base_log + aw_log + co_log
    base_c[j]  = np.exp(base_log         * y_std + y_mean)
    aw_c[j]    = np.exp(total_log         * y_std + y_mean) - np.exp((base_log + co_log) * y_std + y_mean)
    co_c[j]    = np.exp(total_log         * y_std + y_mean) - np.exp((base_log + aw_log) * y_std + y_mean)

df_tr_d = df_train.reset_index(drop=True)
years   = [2023, 2024, 2025]

print("\n  Annual Revenue Decomposition:")
print(f"  {'Year':>6} {'Actual':>10} {'Baseline':>12} {'Awareness':>12} {'Consid.':>12} "
      f"{'Aw ROI':>8} {'Co ROI':>8}")
print("  " + "-" * 74)

decomp_rows = []
for yr in years:
    mask  = (df_tr_d["year"] == yr).values
    actual = df_tr_d.loc[mask, "LBE_Revenue"].sum()
    aw_sp  = df_tr_d.loc[mask, "Awareness"].sum()
    co_sp  = df_tr_d.loc[mask, "Consideration"].sum()
    aw_r   = aw_c[:,  mask].sum(axis=1)
    co_r   = co_c[:,  mask].sum(axis=1)
    base_r = base_c[:, mask].sum(axis=1)
    aw_roi = (aw_r / aw_sp).mean()
    co_roi = (co_r / co_sp).mean()
    aw_roi_ci  = np.percentile(aw_r / aw_sp, [10, 90])
    co_roi_ci  = np.percentile(co_r / co_sp, [10, 90])
    decomp_rows.append(dict(year=yr, actual=actual, baseline=base_r.mean(),
                            awareness=aw_r.mean(), consideration=co_r.mean(),
                            aw_sp=aw_sp, co_sp=co_sp,
                            aw_roi=aw_roi, co_roi=co_roi,
                            aw_roi_lo=aw_roi_ci[0], aw_roi_hi=aw_roi_ci[1],
                            co_roi_lo=co_roi_ci[0], co_roi_hi=co_roi_ci[1],
                            aw_r=aw_r, co_r=co_r))
    print(f"  {yr:>6} ${actual/1e6:>8.1f}M ${base_r.mean()/1e6:>10.1f}M "
          f"${aw_r.mean()/1e6:>10.1f}M ${co_r.mean()/1e6:>10.1f}M "
          f"{aw_roi:>7.2f}x {co_roi:>7.2f}x")

In [0]:
# Plot 7: Decomposition stacked bar + ROI
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("LBE MMM — Revenue Decomposition & ROI (2023–2025)",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

ax = axes[0]
base_vals = [r["baseline"] / 1e6 for r in decomp_rows]
aw_vals   = [r["awareness"] / 1e6 for r in decomp_rows]
co_vals   = [r["consideration"] / 1e6 for r in decomp_rows]
act_vals  = [r["actual"] / 1e6 for r in decomp_rows]
x = np.arange(len(years)); w = 0.55
ax.bar(x, base_vals, w, label="Baseline",      color=CPX_LGREY, edgecolor=CPX_DGREY)
ax.bar(x, aw_vals,   w, bottom=base_vals,       label="Awareness",      color=CPX_NAVY,  alpha=0.85)
ax.bar(x, co_vals,   w, bottom=np.array(base_vals) + np.array(aw_vals),
       label="Consideration", color=CPX_GOLD,  alpha=0.85)
ax.plot(x, act_vals, "D", color=CPX_RED, ms=9, zorder=5, label="Actual")
ax.set_xticks(x); ax.set_xticklabels(years)
ax.set_ylabel("Revenue ($M)"); ax.set_title("Annual Revenue Attribution"); ax.legend()
for i, total in enumerate(np.array(base_vals) + np.array(aw_vals) + np.array(co_vals)):
    ax.text(i, total + 0.5, f"${total:.0f}M", ha="center", fontsize=8, color=CPX_DGREY)

ax = axes[1]
x = np.arange(len(years)); w = 0.35
for i, row in enumerate(decomp_rows):
    ax.bar(i - w/2, row["aw_roi"], w, color=CPX_NAVY, alpha=0.85,
           label="Awareness ROI" if i == 0 else "")
    ax.errorbar(i - w/2, row["aw_roi"],
                yerr=[[row["aw_roi"] - row["aw_roi_lo"]], [row["aw_roi_hi"] - row["aw_roi"]]],
                fmt="none", color="black", capsize=4, lw=1.5)
    ax.bar(i + w/2, row["co_roi"], w, color=CPX_GOLD, alpha=0.85,
           label="Consideration ROI" if i == 0 else "")
    ax.errorbar(i + w/2, row["co_roi"],
                yerr=[[row["co_roi"] - row["co_roi_lo"]], [row["co_roi_hi"] - row["co_roi"]]],
                fmt="none", color="black", capsize=4, lw=1.5)
    ax.text(i - w/2, row["aw_roi"] + 0.1, f"{row['aw_roi']:.1f}x",
            ha="center", fontsize=8, color=CPX_NAVY, fontweight="bold")
    ax.text(i + w/2, max(row["co_roi"], 0.05) + 0.05, f"{row['co_roi']:.2f}x",
            ha="center", fontsize=8, color=CPX_DGREY)
ax.axhline(1, color=CPX_RED, lw=1.5, ls="--", label="Break-even (1x)")
ax.set_xticks(x); ax.set_xticklabels(years)
ax.set_ylabel("ROI (Revenue / Spend)")
ax.set_title("Awareness vs Consideration ROI | Error bars = 80% CI")
ax.legend()

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "07_decomp_roi.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"      Saved: {OUTPUT_DIR}/07_decomp_roi.png")


In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 7. COUNTERFACTUAL ANALYSIS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("\n[6/7] Counterfactual scenarios...")

def scenario_rev(aw_raw: np.ndarray, co_raw: np.ndarray, n_samples: int = 800) -> np.ndarray:
    idx3  = np.random.choice(n_draws, n_samples, replace=False)
    preds = np.zeros((n_samples, len(aw_raw)))
    for j, i in enumerate(idx3):
        aw_ads = adstock_np(aw_raw, float(draws["aw_decay"][i]))
        co_ads = adstock_np(co_raw, float(draws["co_decay"][i]))
        aw_sat = np.log1p(aw_ads * float(draws["aw_scale"][i]))
        co_sat = np.log1p(co_ads * float(draws["co_scale"][i]))
        cal_eff = tr["cal"] @ bc[i]
        mu = (float(draws["intercept"][i])
              + float(draws["beta_aw"][i])    * aw_sat
              + float(draws["beta_co"][i])    * co_sat
              + float(draws["beta_trend"][i]) * tr["t"]
              + float(draws["beta_temp"][i])  * tr["temp"]
              + cal_eff)
        preds[j] = np.exp(mu * y_std + y_mean)
    return preds

np.random.seed(RANDOM_SEED + 10)
aw_a = tr["aw"]; co_a = tr["co"]
sc_names = [
    "Actual",
    "Zero media",
    "Zero awareness",
    "Zero consideration",
    "Double awareness",
    "Double consideration",
    "Shift all consid → aware",
    "Shift 50% consid → aware",
    "+50% awareness only",
]
sc_inputs = [
    (aw_a,              co_a),
    (np.zeros_like(aw_a), np.zeros_like(co_a)),
    (np.zeros_like(aw_a), co_a),
    (aw_a,              np.zeros_like(co_a)),
    (aw_a * 2,          co_a),
    (aw_a,              co_a * 2),
    (aw_a + co_a,       np.zeros_like(co_a)),
    (aw_a + co_a * 0.5, co_a * 0.5),
    (aw_a * 1.5,        co_a),
]
sc_draws = {n: scenario_rev(aw, co, 800).sum(axis=1)
            for n, (aw, co) in zip(sc_names, sc_inputs)}
act_sc = sc_draws["Actual"]

print("\n  3-Year Counterfactual Summary:")
print(f"  {'Scenario':<30} {'Revenue':>9} {'vs Actual':>11} {'80% CI':>22}")
print("  " + "-" * 75)
for name, d_sc in sc_draws.items():
    m    = d_sc.mean()
    diff = (d_sc - act_sc).mean()
    lo   = np.percentile(d_sc - act_sc, 10)
    hi   = np.percentile(d_sc - act_sc, 95)
    flag = " ◄" if name == "Actual" else ""
    ci   = f"[{lo/1e6:+.1f}M to {hi/1e6:+.1f}M]" if name != "Actual" else ""
    print(f"  {name:<30} ${m/1e6:>7.1f}M  {diff/1e6:>+9.1f}M  {ci}{flag}")

In [0]:
# Plot 8: Counterfactual chart
fig, ax = plt.subplots(figsize=(14, 7))
fig.suptitle("LBE MMM — 3-Year Counterfactual Scenarios (2023–2025)",
             fontsize=13, fontweight="bold", color=CPX_NAVY)

names = list(sc_draws.keys())
means_cf = [(sc_draws[n] - act_sc).mean() / 1e6 for n in names]
ci_lo    = [np.percentile(sc_draws[n] - act_sc, 10) / 1e6 for n in names]
ci_hi    = [np.percentile(sc_draws[n] - act_sc, 90) / 1e6 for n in names]

colors_cf = []
for i, (n, m) in enumerate(zip(names, means_cf)):
    if n == "Actual":         colors_cf.append(CPX_DGREY)
    elif m > 3:               colors_cf.append(CPX_NAVY)
    elif m > 0:               colors_cf.append("#5B8DB8")
    else:                     colors_cf.append(CPX_RED)

y_pos = np.arange(len(names))
bars  = ax.barh(y_pos, means_cf, color=colors_cf, alpha=0.85, height=0.6)
ax.errorbar(means_cf, y_pos,
            xerr=[np.array(means_cf) - np.array(ci_lo),
                  np.array(ci_hi) - np.array(means_cf)],
            fmt="none", color=CPX_DGREY, capsize=4, lw=1.5)
ax.axvline(0, color="black", lw=1)

for i, (bar, m) in enumerate(zip(bars, means_cf)):
    if abs(m) > 0.5:
        xpos = m + (0.2 if m >= 0 else -0.2)
        align = "left" if m >= 0 else "right"
        ax.text(xpos, bar.get_y() + bar.get_height() / 2,
                f"{m:+.1f}M", va="center", ha=align, fontsize=8,
                fontweight="bold", color=CPX_DGREY)

ax.set_yticks(y_pos); ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel("Revenue vs Actual ($M)")
ax.set_title("Incremental Revenue vs Actual  |  Error bars = 80% CI\n"
             "Note: 'Actual' bar = 0 by construction")

plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "08_counterfactual.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"      Saved: {OUTPUT_DIR}/08_counterfactual.png")


In [0]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 8. SAVE RESULTS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f"\n[7/7] Saving model state to {RESULTS_PKL}...")

results = {
    "idata":       idata,
    "draws":       draws,
    "bc":          bc,
    "n_draws":     n_draws,
    "tr":          tr, "va": va,
    "df_train":    df_train, "df_val": df_val,
    "y_mean":      y_mean, "y_std": y_std,
    "aw_max":      aw_max, "co_max": co_max,
    "t_max":       t_max, "temp_mu": temp_mu, "temp_sd": temp_sd,
    "CAL_COLS":    CAL_COLS, "CAL_MU": CAL_MU, "CAL_SIGMA": CAL_SIGMA,
    "metrics":     dict(train_mape=train_mape, val_mape=val_mape,
                        train_r2=train_r2, val_r2=val_r2),
    "decomp":      decomp_rows,
    "counterfactuals": {n: {"mean": sc_draws[n].mean(),
                            "ci80_lo": np.percentile(sc_draws[n], 10),
                            "ci80_hi": np.percentile(sc_draws[n], 90)}
                        for n in sc_names},
}

with open(RESULTS_PKL, "wb") as f:
    pickle.dump(results, f)

print("\n" + "=" * 65)
print("  COMPLETE")
print("=" * 65)
print(f"\n  Train MAPE : {train_mape:.1f}%   |   Train R² : {train_r2:.3f}")
print(f"  Val   MAPE : {val_mape:.1f}%   |   Val   R² : {val_r2:.3f}")
print(f"  Divergences: {divs}")
print(f"\n  Key findings:")
mask_3yr = np.ones(T, dtype=bool)
aw_3yr = aw_c.sum(axis=1); co_3yr = co_c.sum(axis=1)
aw_sp_3yr = df_train["Awareness"].sum(); co_sp_3yr = df_train["Consideration"].sum()
print(f"    Awareness  ROI (3yr): {(aw_3yr/aw_sp_3yr).mean():.2f}x  "
      f"[80% CI: {np.percentile(aw_3yr/aw_sp_3yr,10):.2f}x – "
      f"{np.percentile(aw_3yr/aw_sp_3yr,90):.2f}x]")
print(f"    Consideration ROI (3yr): {(co_3yr/co_sp_3yr).mean():.2f}x  "
      f"[80% CI: {np.percentile(co_3yr/co_sp_3yr,10):.2f}x – "
      f"{np.percentile(co_3yr/co_sp_3yr,90):.2f}x]")
print(f"\n  Adstock decay:")
print(f"    Awareness:     {draws['aw_decay'].mean():.2f}  "
      f"(response peaks ~4–6 weeks after spend)")
print(f"    Consideration: {draws['co_decay'].mean():.2f}  "
      f"(fast decay, near-zero lagged signal)")
print(f"\n  Plots saved to: {OUTPUT_DIR}/")
print(f"  Model state:    {RESULTS_PKL}")
print("=" * 65)
